In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE3_DIR = f"{PROJECT_DIR}/phase3_outputs"
PHASE4_DIR = f"{PROJECT_DIR}/phase4_outputs"
os.makedirs(PHASE4_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
test_closed_records = load_jsonl(f"{PHASE3_DIR}/test_closed.jsonl")
test_open_records = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")

n_train_poets = len(set(r["poet"] for r in train_records))
print(f"train: {len(train_records)} نمونه / {n_train_poets} شاعر")
assert n_train_poets == 150, (
    f"❌ انتظار داشتیم دقیقاً ۱۵۰ شاعر در train، شد {n_train_poets}. قبل از ادامه PHASE3_DIR را بررسی کنید."
)
print(f"test_closed: {len(test_closed_records)} نمونه")
print(f"test_open:   {len(test_open_records)} نمونه")

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
train_poem_ids = [r["poem_id"] for r in train_records]


In [ ]:
import numpy as np

train_embeddings = np.load(f"{PHASE3_DIR}/train_embeddings_byt5base.npy")
test_closed_embeddings = np.load(f"{PHASE3_DIR}/test_closed_embeddings_byt5base.npy")
test_open_embeddings = np.load(f"{PHASE3_DIR}/test_open_embeddings_byt5base.npy")

print("شکلِ embeddingِ train:", train_embeddings.shape)
print("شکلِ embeddingِ test_closed:", test_closed_embeddings.shape)
print("شکلِ embeddingِ test_open:", test_open_embeddings.shape)

assert train_embeddings.shape[0] == len(train_records), "❌ تعدادِ ردیف‌هایِ embeddingِ train با train_records یکی نیست"
assert test_closed_embeddings.shape[0] == len(test_closed_records), "❌ تعدادِ ردیف‌هایِ embeddingِ test_closed یکی نیست"
assert test_open_embeddings.shape[0] == len(test_open_records), "❌ تعدادِ ردیف‌هایِ embeddingِ test_open یکی نیست"
assert np.isnan(train_embeddings).any() == False
assert np.isnan(test_closed_embeddings).any() == False
assert np.isnan(test_open_embeddings).any() == False
print("✅ هیچ NaN در هیچ‌کدام از سه embedding وجود ندارد.")


In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=50000, sublinear_tf=True, dtype=np.float32)
vectorizer.fit(train_texts)
print(f"TF-IDF رویِ Train فیت شد. اندازه‌یِ واژگان: {len(vectorizer.vocabulary_)}  ({time.time()-t0:.1f}s)")

train_tfidf = vectorizer.transform(train_texts)
print("شکلِ TF-IDFِ train (galleryِ بسته):", train_tfidf.shape, f"  ({time.time()-t0:.1f}s)")


In [ ]:
from collections import defaultdict

def group_by_k(records, embeddings):
    by_k = defaultdict(lambda: {"records": [], "embeddings": []})
    for r, e in zip(records, embeddings):
        by_k[r["k"]]["records"].append(r)
        by_k[r["k"]]["embeddings"].append(e)
    for k in by_k:
        by_k[k]["embeddings"] = np.stack(by_k[k]["embeddings"])
    return dict(sorted(by_k.items()))

test_closed_by_k = group_by_k(test_closed_records, test_closed_embeddings)
test_open_by_k = group_by_k(test_open_records, test_open_embeddings)
K_VALUES = [1, 2, 4, 6]

print("=== Test-Closed بر اساسِ k ===")
for k in K_VALUES:
    print(f"  k={k}: {len(test_closed_by_k[k]['records']):,} نمونه")
print("\n=== Test-Open بر اساسِ k ===")
for k in K_VALUES:
    print(f"  k={k}: {len(test_open_by_k[k]['records']):,} نمونه")

closed_tfidf_by_k, open_tfidf_by_k = {}, {}
for k in K_VALUES:
    t0 = time.time()
    closed_tfidf_by_k[k] = vectorizer.transform([r["text"] for r in test_closed_by_k[k]["records"]])
    open_tfidf_by_k[k] = vectorizer.transform([r["text"] for r in test_open_by_k[k]["records"]])
    print(f"k={k}: TF-IDF closed {closed_tfidf_by_k[k].shape}, open {open_tfidf_by_k[k].shape}  ({time.time()-t0:.1f}s)")


In [ ]:
import random

def stratified_dev_split(poets, dev_frac=0.15, seed=42):
    rng = random.Random(seed)
    by_poet = defaultdict(list)
    for i, p in enumerate(poets):
        by_poet[p].append(i)
    dev_idx = []
    for p, idxs in by_poet.items():
        idxs = idxs[:]
        rng.shuffle(idxs)
        n_dev = max(1, round(len(idxs) * dev_frac))
        dev_idx.extend(idxs[:n_dev])
    return sorted(dev_idx)

dev_indices = {}
for k in K_VALUES:
    closed_poets_k = [r["poet"] for r in test_closed_by_k[k]["records"]]
    open_poets_k = [r["poet"] for r in test_open_by_k[k]["records"]]
    dev_indices[("closed", k)] = stratified_dev_split(closed_poets_k)
    dev_indices[("open", k)] = stratified_dev_split(open_poets_k)
    print(f"k={k} | dev: closed={len(dev_indices[('closed', k)])}/{len(closed_poets_k)} "
          f"| open={len(dev_indices[('open', k)])}/{len(open_poets_k)}")


In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize as sk_normalize
from collections import defaultdict

def l2_normalize(mat):
    if sp.issparse(mat):
        return sk_normalize(mat, norm="l2", axis=1)
    mat = np.asarray(mat, dtype=np.float32)
    return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)

def compute_sims(batch_q, g):
    sims = batch_q @ g.T
    if sp.issparse(sims):
        sims = sims.toarray()
    return np.asarray(sims, dtype=np.float32)

def evaluate_retrieval(query_vecs, query_poets, query_poem_ids,
                        gallery_vecs, gallery_poets, gallery_poem_ids,
                        ks=(1, 5, 10), exclude_same_poem=False, batch_size=500):
    q = l2_normalize(query_vecs); g = l2_normalize(gallery_vecs)
    query_poets = np.asarray(query_poets); gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids); gallery_poem_ids = np.asarray(gallery_poem_ids)
    max_k = max(ks); n_gallery = g.shape[0]
    recall_hits = {k: 0 for k in ks}; mrr_total = 0.0; valid_queries = 0
    n_queries = q.shape[0]
    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_q = q[start:end]
        sims = compute_sims(batch_q, g)
        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            sims = np.where(same_poem_mask, -np.inf, sims)
        k_eff = min(max_k, n_gallery - 1) if exclude_same_poem else min(max_k, n_gallery)
        top_idx = np.argpartition(-sims, kth=k_eff - 1, axis=1)[:, :k_eff]
        row_idx = np.arange(top_idx.shape[0])[:, None]
        top_sims = sims[row_idx, top_idx]
        order = np.argsort(-top_sims, axis=1)
        top_idx_sorted = top_idx[row_idx, order]
        top_sims_sorted = top_sims[row_idx, order]
        batch_poets = query_poets[start:end]
        for i in range(end - start):
            if np.all(np.isneginf(top_sims_sorted[i])): continue
            valid_queries += 1
            retrieved_poets = gallery_poets[top_idx_sorted[i]]
            hit_positions = np.where(retrieved_poets == batch_poets[i])[0]
            if len(hit_positions) > 0:
                rank = hit_positions[0] + 1
                mrr_total += 1.0 / rank
                for k in ks:
                    if rank <= k: recall_hits[k] += 1
    out = {"n_queries": valid_queries, "mrr": mrr_total / valid_queries if valid_queries else float("nan")}
    for k in ks: out[f"recall@{k}"] = recall_hits[k] / valid_queries if valid_queries else float("nan")
    return out

def top_c_ranks(sims, c):
    n_gallery = sims.shape[1]
    c_eff = min(c, n_gallery)
    top_idx = np.argpartition(-sims, kth=c_eff - 1, axis=1)[:, :c_eff]
    row_idx = np.arange(sims.shape[0])[:, None]
    top_sims = sims[row_idx, top_idx]
    order = np.argsort(-top_sims, axis=1)
    top_idx_sorted = top_idx[row_idx, order]
    ranks = np.tile(np.arange(1, c_eff + 1), (sims.shape[0], 1))
    return top_idx_sorted, ranks

def evaluate_retrieval_rrf(dense_q, dense_g, sparse_q, sparse_g,
                            query_poets, query_poem_ids, gallery_poets, gallery_poem_ids,
                            mu, ks=(1, 5, 10), exclude_same_poem=False, batch_size=200, c_depth=500):
    dq = l2_normalize(dense_q); dg = l2_normalize(dense_g)
    sq = l2_normalize(sparse_q); sg = l2_normalize(sparse_g)
    query_poets = np.asarray(query_poets); gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids); gallery_poem_ids = np.asarray(gallery_poem_ids)
    n_queries = dq.shape[0]; max_k = max(ks)
    recall_hits = {k: 0 for k in ks}; mrr_total = 0.0; valid_queries = 0
    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        bdq = dq[start:end]; bsq = sq[start:end]
        dense_sims = compute_sims(bdq, dg)
        sparse_sims = compute_sims(bsq, sg)
        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            dense_sims = np.where(same_poem_mask, -np.inf, dense_sims)
            sparse_sims = np.where(same_poem_mask, -np.inf, sparse_sims)
        dense_idx, dense_rank = top_c_ranks(dense_sims, c_depth)
        sparse_idx, sparse_rank = top_c_ranks(sparse_sims, c_depth)
        b = end - start
        batch_poets = query_poets[start:end]
        for i in range(b):
            scores = defaultdict(float)
            for rank, gi in zip(dense_rank[i].tolist(), dense_idx[i].tolist()):
                scores[gi] += 1.0 / (60 + rank)
            for rank, gi in zip(sparse_rank[i].tolist(), sparse_idx[i].tolist()):
                scores[gi] += mu / (60 + rank)
            if not scores: continue
            sorted_gi = sorted(scores.keys(), key=lambda gi: -scores[gi])[:max_k]
            valid_queries += 1
            retrieved_poets = gallery_poets[sorted_gi]
            hit_positions = np.where(retrieved_poets == batch_poets[i])[0]
            if len(hit_positions) > 0:
                rank = hit_positions[0] + 1
                mrr_total += 1.0 / rank
                for k in ks:
                    if rank <= k: recall_hits[k] += 1
    out = {"n_queries": valid_queries, "mrr": mrr_total / valid_queries if valid_queries else float("nan")}
    for k in ks: out[f"recall@{k}"] = recall_hits[k] / valid_queries if valid_queries else float("nan")
    return out

print("✅ توابعِ evaluate_retrieval (دقیق) و evaluate_retrieval_rrf (تقریبِ c_depth) تعریف شدند.")


In [ ]:
import time as _time
baseline_results = {}
for k in K_VALUES:
    d = test_closed_by_k[k]
    qp = [r["poet"] for r in d["records"]]; qpid = [r["poem_id"] for r in d["records"]]
    _t0 = _time.time()
    baseline_results[("closed", k, "pacl")] = evaluate_retrieval(
        d["embeddings"], qp, qpid, train_embeddings, train_poets, train_poem_ids,
        exclude_same_poem=False)
    print(f"  [k={k}] closed-PACL  در {_time.time()-_t0:.1f}s تمام شد")
    _t0 = _time.time()
    baseline_results[("closed", k, "tfidf")] = evaluate_retrieval(
        closed_tfidf_by_k[k], qp, qpid, train_tfidf, train_poets, train_poem_ids,
        exclude_same_poem=False)
    print(f"  [k={k}] closed-TFIDF در {_time.time()-_t0:.1f}s تمام شد")
    d = test_open_by_k[k]
    qp = [r["poet"] for r in d["records"]]; qpid = [r["poem_id"] for r in d["records"]]
    _t0 = _time.time()
    baseline_results[("open", k, "pacl")] = evaluate_retrieval(
        d["embeddings"], qp, qpid, d["embeddings"], qp, qpid, exclude_same_poem=True)
    print(f"  [k={k}] open-PACL    در {_time.time()-_t0:.1f}s تمام شد")
    _t0 = _time.time()
    baseline_results[("open", k, "tfidf")] = evaluate_retrieval(
        open_tfidf_by_k[k], qp, qpid, open_tfidf_by_k[k], qp, qpid, exclude_same_poem=True)
    print(f"  [k={k}] open-TFIDF   در {_time.time()-_t0:.1f}s تمام شد")
    print(f"k={k} | closed PACL R@1={baseline_results[('closed',k,'pacl')]['recall@1']:.4f} "
          f"TF-IDF R@1={baseline_results[('closed',k,'tfidf')]['recall@1']:.4f} "
          f"| open PACL R@1={baseline_results[('open',k,'pacl')]['recall@1']:.4f} "
          f"TF-IDF R@1={baseline_results[('open',k,'tfidf')]['recall@1']:.4f}")


In [ ]:
MU_CANDIDATES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]
mu_sweep_results = defaultdict(dict)

for k in K_VALUES:
    for scenario in ["closed", "open"]:
        dev_idx = dev_indices[(scenario, k)]
        if scenario == "closed":
            d = test_closed_by_k[k]
            dense_g, sparse_g = train_embeddings, train_tfidf
            gallery_poets, gallery_poem_ids = train_poets, train_poem_ids
            sparse_q_full = closed_tfidf_by_k[k]
            exclude = False
        else:
            d = test_open_by_k[k]
            dense_g = d["embeddings"]
            sparse_g = open_tfidf_by_k[k]
            gallery_poets = [r["poet"] for r in d["records"]]
            gallery_poem_ids = [r["poem_id"] for r in d["records"]]
            sparse_q_full = open_tfidf_by_k[k]
            exclude = True
        dev_dense_q = d["embeddings"][dev_idx]
        dev_sparse_q = sparse_q_full[dev_idx]
        dev_poets = [d["records"][i]["poet"] for i in dev_idx]
        dev_poem_ids = [d["records"][i]["poem_id"] for i in dev_idx]
        for mu in MU_CANDIDATES:
            res = evaluate_retrieval_rrf(
                dev_dense_q, dense_g, dev_sparse_q, sparse_g,
                dev_poets, dev_poem_ids, gallery_poets, gallery_poem_ids,
                mu=mu, exclude_same_poem=exclude, batch_size=200, c_depth=300,
            )
            mu_sweep_results[mu][(scenario, k)] = res["recall@1"]
        print(f"✅ جاروبِ μ برایِ {scenario} k={k} تمام شد.")

best_mu, best_avg = None, -1
for mu in MU_CANDIDATES:
    avg = float(np.mean(list(mu_sweep_results[mu].values())))
    print(f"μ={mu}: میانگینِ Recall@1 رویِ dev (۸ ترکیب) = {avg:.4f}")
    if avg > best_avg:
        best_avg, best_mu = avg, mu

print(f"\n🏆 بهترین μِ سراسری: {best_mu} (میانگینِ Recall@1 رویِ dev = {best_avg:.4f})")


In [ ]:
rrf_results = {}
for k in K_VALUES:
    d = test_closed_by_k[k]
    qp = [r["poet"] for r in d["records"]]; qpid = [r["poem_id"] for r in d["records"]]
    rrf_results[("closed", k)] = evaluate_retrieval_rrf(
        d["embeddings"], train_embeddings, closed_tfidf_by_k[k], train_tfidf,
        qp, qpid, train_poets, train_poem_ids,
        mu=best_mu, exclude_same_poem=False, batch_size=200, c_depth=500)
    d = test_open_by_k[k]
    qp = [r["poet"] for r in d["records"]]; qpid = [r["poem_id"] for r in d["records"]]
    rrf_results[("open", k)] = evaluate_retrieval_rrf(
        d["embeddings"], d["embeddings"], open_tfidf_by_k[k], open_tfidf_by_k[k],
        qp, qpid, qp, qpid,
        mu=best_mu, exclude_same_poem=True, batch_size=200, c_depth=500)
    print(f"k={k} | closed RRF R@1={rrf_results[('closed',k)]['recall@1']:.4f} "
          f"| open RRF R@1={rrf_results[('open',k)]['recall@1']:.4f}")

print("\n=== جدولِ مقایسه‌یِ نهایی (Recall@1) ===")
print(f"{'scenario':<8}{'k':<4}{'PACL':<10}{'TF-IDF':<10}{'RRF-fused':<12}{'بهبود نسبت به بهترینِ قبلی'}")
for scenario in ["closed", "open"]:
    for k in K_VALUES:
        pacl = baseline_results[(scenario, k, "pacl")]["recall@1"]
        tfidf = baseline_results[(scenario, k, "tfidf")]["recall@1"]
        rrf = rrf_results[(scenario, k)]["recall@1"]
        best_prev = max(pacl, tfidf)
        delta = rrf - best_prev
        print(f"{scenario:<8}{k:<4}{pacl:<10.4f}{tfidf:<10.4f}{rrf:<12.4f}{delta:+.4f}")

open_k1_r1 = rrf_results[("open", 1)]["recall@1"]
open_k6_r1 = rrf_results[("open", 6)]["recall@1"]
print(f"\n🎯 هدفِ  — Open k=1: Recall@1 ≥ 0.385  | نتیجه: {open_k1_r1:.4f}  "
      f"{'✅ محقق شد' if open_k1_r1 >= 0.385 else '❌ محقق نشد'}")
print(f"🎯 هدفِ  — Open k=6: Recall@1 ≥ 0.635  | نتیجه: {open_k6_r1:.4f}  "
      f"{'✅ محقق شد' if open_k6_r1 >= 0.635 else '❌ محقق نشد'}")


In [ ]:
import json as _json

def pack(res_dict, keys3=False):
    out = {}
    for key, res in res_dict.items():
        k_str = "_".join(str(x) for x in key)
        out[k_str] = res
    return out

summary = {
    "method": "RRF (Reciprocal Rank Fusion) — dense(ByT5 epoch9) + sparse(TF-IDF char_wb)",
    "formula": "S_RRF(q,d) = 1/(60+rank_dense) + mu/(60+rank_sparse)",
    "c_depth_final_eval": 500,
    "c_depth_mu_sweep": 300,
    "dev_frac_for_mu_sweep": 0.15,
    "mu_candidates": MU_CANDIDATES,
    "best_mu": best_mu,
    "best_mu_dev_avg_recall1": best_avg,
    "baseline_results": pack(baseline_results),
    "rrf_results": pack(rrf_results),
    "professor_targets": {
        "open_k1_recall1_target": 0.385,
        "open_k1_recall1_actual": open_k1_r1,
        "open_k6_recall1_target": 0.635,
        "open_k6_recall1_actual": open_k6_r1,
    },
}

out_path = f"{PHASE4_DIR}/step1_rrf_fusion_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    _json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"✅ ذخیره شد در {out_path}")
